In [11]:
# Sam Brown
# sam_brown@mines.edu
# June19
# Goal: Preprocess data and create dataframe for analysis of long-term slip patterns

# Get tide data for entire timeframe using average gz coords (no tide in la stations but timing is still relevant)

import sys
sys.path.append("../")

import my_lib.funcs
import Stations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# NOTES: use GZ05 for coordinates to retrieve tide data.

# For this dataset we are focusing on tidal modulation. For each event we will need tide height, tide derivative, form factor, 
# time since last event, slip size (standardized for each station and averaged), high or low tide event, and some indicator
# to signal whether the event is following a skipped low/high tide event.

In [12]:
# Load the paths

# df_2008 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2008_2008Events2stas")
# df_2009 = my_lib.funcs.load_evt("/Users/sambrown/Documents/SURF/Events/2009_2009Events2stas")
df_2010 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2010_2010Events2stas")
df_2011 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas")
df_2012 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2012_2012Events2stas")
df_2013 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas")
df_2014 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2014_2014Events2stas")
df_2015 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2015_2015Events2stas")
df_2016 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2016_2016Events2stas")
df_2017 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2017_2017Events2stas")
df_2018 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2018_2018Events2stas")
df_2019 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2019_2019Events2stas")

# One Large list
all_dfs = (
    df_2010 + df_2011 + df_2012 + df_2013 +
    df_2014 + df_2015 + df_2016 + df_2017 + df_2018 + df_2019
)


In [4]:
df_2011[0]['gz05y'].head()

0   -604863.615642
1   -604863.615021
2   -604863.614049
3   -604863.613840
4   -604863.619055
Name: gz05y, dtype: float64

In [14]:
# Preprocess
clean_df = my_lib.funcs.extract_event_features(all_dfs)

In [18]:
clean_df[0].head()

,station,pre-slip_area,slip_severity,peak_time,total_delta,start_time
0,la01x,86.120885,6.613400e-07,4755.0,0.319709,2010-11-18 23:03:00
1,la02x,78.146228,5.853301e-07,4860.0,0.296645,2010-11-18 23:03:00
2,la04x,140.100307,6.513987e-07,4845.0,0.295982,2010-11-18 23:03:00
3,la05x,89.826900,3.453581e-07,4785.0,0.208539,2010-11-18 23:03:00
4,la06x,72.676269,9.571907e-07,4785.0,0.335858,2010-11-18 23:03:00


In [20]:
standards = pd.read_csv('../station_standards.csv')

In [22]:
standards.head()

,Station,pre-slip_area,pre-slip_area_sd,slip_severity,slip_severity_sd,slip_size,slip_size_sd,x_cor,y_cor
0,la01,120.336973,48.471627,1.026324e-06,3.246978e-07,0.365944,0.057366,-280745.085119,-559652.084157
1,la02,107.824098,77.972644,1.073755e-06,3.649910e-07,0.377533,0.061455,-276605.572091,-561027.240679
2,la03,117.900913,93.697779,9.407593e-07,2.875423e-07,0.362463,0.053384,-274115.321381,-565881.245886
3,la04,89.389549,87.318524,1.017707e-06,3.423616e-07,0.391973,0.065167,-255279.728700,-568381.970285
4,la05,90.774336,96.847303,8.644231e-07,3.803968e-07,0.358512,0.094409,-243775.086715,-563142.679534


In [24]:
# Loop through each event. Loop through each station in events, append the standardized delta to the list then average it.
# add start time to data frame

# Initialize dataframe
net_df = pd.DataFrame(columns = [ 'tide_deriv', 'form_fac', 'time_since', 'slip_size_standardized', 'high_t_evt', 'start_time', 'sev_stds', 'pre-s_stds']) # Tide height will be added by merge

for event in clean_df:

    # Initialize list for slip sizes
    slip_deltas = []
    slip_sevs = []
    slip_pre = []

    # Loop through rows
    for i, row in event.iterrows():
        station = row['station'][:4]
        if station == 'slw1' or station =='ws04' or station =='ws05':
            continue

        row_sch = standards[standards['Station'] == station]

        # if row_sch.empty:
        #     raise ValueError(f"Station not found in standards: '{station}'")
        
        # Standardization size
        station_mean = row_sch['slip_size'].iloc[0]
        station_sd = row_sch['slip_size_sd'].iloc[0]
        standardized_val = (row['total_delta'] - station_mean) / station_sd
        # print(f"{row['total_delta']}, {station}, {station_mean}, {station_sd}")

        # Standardization slip severity
        station_sev_u = row_sch['slip_severity'].iloc[0]
        station_sev_o = row_sch['slip_severity_sd'].iloc[0]

        standardized_val_sev = (row['slip_severity'] - station_sev_u) / station_sev_o

        # Standardization pre-slip area
        station_slip_u = row_sch['pre-slip_area'].iloc[0]
        station_slip_o = row_sch['pre-slip_area_sd'].iloc[0]

        standardized_val_slip = (row['pre-slip_area'] - station_slip_u) / station_slip_o        

        slip_deltas.append(standardized_val)
        slip_sevs.append(standardized_val_sev)
        slip_pre.append(standardized_val_slip)
        

    net_df.loc[len(net_df)] = {
        "slip_size_standardized": sum(slip_deltas) / len(slip_deltas),
        "start_time": event.at[0, 'start_time'],
        "sev_stds": sum(slip_sevs) / len(slip_sevs),
        "pre-s_stds": sum(slip_pre) / len(slip_pre)
    }

In [26]:
# Load tide Data
# NOTE: previously averaged coors of gz stations and had missing vals; will use gz05 now

#Average coordinates for gz stations (source code in severity_class nb)
# x_cor = -168955.1491394913 
# y_cor = -599694.5432784811

# GZ05 coors
x_cor = -155992.359664
y_cor = -604863.615642

tide_df = my_lib.funcs.get_tide_height(4380, x_cor, y_cor, "2008-01-01 00:00:00") # 12 years worth of data

Elapsed time: 44.83895397186279 seconds


In [27]:
# Organize events by time
net_df = net_df.sort_values('start_time')

In [28]:
# Calculate time since in minutes
net_df['time_since'] = net_df['start_time'].diff().dt.total_seconds() / 60

In [29]:
# Need to get start times down to minutes
tide_df['time'] = tide_df['time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))
net_df['start_time'] = net_df['start_time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))

In [30]:
# Insert Tide values
net_df['start_time'] = pd.to_datetime(net_df['start_time'])
tide_df['time'] = pd.to_datetime(tide_df['time'])

# Merge tide height into net_df based on matching timestamps
merged_df = pd.merge(net_df, tide_df[['time', 'tide_height']], 
                     left_on='start_time', right_on='time', how='left')

# Drop extra 'time' column if you want
merged_df = merged_df.drop(columns=['time'])

In [31]:
# Insert tide derivatives into data
tide_d = my_lib.funcs.tide_derivative(tide_df)
for i, row in merged_df.iterrows():
    time = row['start_time']

    index = tide_d[tide_d['time'] == time].index

    if not index.empty:
        idx = index[0]
        merged_df.at[i, 'tide_deriv'] = tide_d.at[idx, 'tide_deriv']

In [32]:
merged_df.head(40)

,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,sev_stds,pre-s_stds,tide_height
0,0.024640,NaN,NaN,-1.931350,NaN,2010-01-01 15:25:00,-0.270159,-0.831226,108.956127
1,-0.232954,NaN,512.50,-2.792976,NaN,2010-01-01 23:58:00,-0.668066,-0.739235,-126.423818
2,-0.087155,NaN,1037.75,-0.793920,NaN,2010-01-02 17:15:00,-0.180573,0.594509,94.249899
3,0.138536,NaN,1382.75,-0.838199,NaN,2010-01-03 16:18:00,-0.209510,-0.773764,65.04087
4,-0.484872,NaN,445.00,-1.435594,NaN,2010-01-03 23:43:00,-0.204831,-0.806752,-57.608298
5,0.032944,NaN,912.75,-1.714496,NaN,2010-01-04 14:56:00,-0.485152,-0.800717,23.343183
6,-0.371611,NaN,555.25,-2.816169,NaN,2010-01-05 00:11:00,-0.420283,-0.054552,-34.390384
7,-0.143135,NaN,857.50,-2.154250,NaN,2010-01-05 14:29:00,-0.513920,-0.236468,10.783952
8,-0.168170,NaN,697.75,-1.742904,NaN,2010-01-06 02:06:00,-0.507415,-0.762793,-32.745887
9,-0.251834,NaN,777.50,-2.141397,NaN,2010-01-06 15:04:00,-0.582855,-0.665913,4.320662


In [33]:
# Form factor calculation

form_fac = my_lib.funcs.form_factor_calc(tide_df)

/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF/data_preproc/../my_lib/funcs.py:411: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)


In [34]:
# Add date-only column to form_fac
form_fac['date_only'] = form_fac['dates'].dt.date

# Loop through each row in avg_dat
for i, event in merged_df.iterrows():
    time = event['start_time']
    target_date = time.date()

    # Select all rows with matching date
    rows_date = form_fac[form_fac['date_only'] == target_date]

    # Compute average form factor for that date
    merged_df.at[i, 'form_fac'] = rows_date['form_factors'].mean()

In [35]:
# Encode high tide vs low tide event
merged_df['high_t_evt'] = (merged_df['tide_height'] > 0).astype(int)

In [36]:
merged_df.head(40)

,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,sev_stds,pre-s_stds,tide_height
0,0.024640,3.782138,NaN,-1.931350,1,2010-01-01 15:25:00,-0.270159,-0.831226,108.956127
1,-0.232954,3.782138,512.50,-2.792976,0,2010-01-01 23:58:00,-0.668066,-0.739235,-126.423818
2,-0.087155,3.126144,1037.75,-0.793920,1,2010-01-02 17:15:00,-0.180573,0.594509,94.249899
3,0.138536,2.571349,1382.75,-0.838199,1,2010-01-03 16:18:00,-0.209510,-0.773764,65.04087
4,-0.484872,2.571349,445.00,-1.435594,0,2010-01-03 23:43:00,-0.204831,-0.806752,-57.608298
5,0.032944,2.023008,912.75,-1.714496,1,2010-01-04 14:56:00,-0.485152,-0.800717,23.343183
6,-0.371611,1.364373,555.25,-2.816169,0,2010-01-05 00:11:00,-0.420283,-0.054552,-34.390384
7,-0.143135,1.364373,857.50,-2.154250,1,2010-01-05 14:29:00,-0.513920,-0.236468,10.783952
8,-0.168170,1.127449,697.75,-1.742904,0,2010-01-06 02:06:00,-0.507415,-0.762793,-32.745887
9,-0.251834,1.127449,777.50,-2.141397,1,2010-01-06 15:04:00,-0.582855,-0.665913,4.320662


In [55]:
# We would like to add feature(s) that takes the time period between events and retrieves the form factor for the time period between these events. 
# Two different features, semi diurnal fit and diurnal fit 

# Loop through each event 
for i, evt in merged_df.iterrows():

    # Can't find time before first event
    if i < 1:
        continue
    
    # get start time and duration minutes
    curDate = merged_df.at[i, 'start_time']
    prevDate = merged_df.at[i-1, 'start_time']

    diff = curDate - prevDate
    minutes = diff.total_seconds() / 60.0

    # Call new window function

    # calculate form factor for this time period
    [form_fac, A1, A2, phi1, phi2] = my_lib.funcs.form_factor_window(tide_df, prevDate, minutes)
    # print([form_fac, A1, A2, phi1, phi2])

    # insert the features into the data set for that row
    merged_df.loc[i, 'inter_form_Fac'] = form_fac
    merged_df.loc[i, 'A_diurn'] = A1
    merged_df.loc[i, 'A_semidiurn'] = A2
    # merged_df.loc[i, 'phi1'] = phi1
    # merged_df.loc[i, 'phi2'] = phi2

# Features for a given event will be the coefficients for form factor calc leading up to this event.

In [56]:
# Export to csv file

merged_df.to_csv("/Users/sambrown04/Documents/SURF/Preproc_data/10-18.csv", index = False)